In [1]:
from pathlib import Path
import os
import random
import time
import json
import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.model_selection import train_test_split

print("=" * 70)
print("FASTER R-CNN + RESNET50-FPN EXPERIMENT")
print("=" * 70)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Automatically find dataset root
# ------------------------------------------------------------
possible_roots = [
    Path(
        "/kaggle/input/datasets/smnahian/"
        "low-light-final/vehicle_lowlight_final"
    ),
    Path(
        "/kaggle/input/low-light-final/"
        "vehicle_lowlight_final"
    ),
]

BASE = None
for p in possible_roots:
    if p.exists():
        BASE = p
        break

if BASE is None:
    matches = list(
        Path("/kaggle/input").rglob("vehicle_lowlight_final")
    )
    if len(matches) > 0:
        BASE = matches[0]

if BASE is None:
    raise FileNotFoundError(
        "vehicle_lowlight_final folder could not be found."
    )

ORIGINAL_ROOT = (BASE / "vehicle_dataset" / "Images")
ANNOTATION_ROOT = (BASE / "vehicle_dataset" / "Annotations")
ENHANCED_ROOT = (BASE / "enhanced_vehicle_dataset" / "Images")

print("\nDataset root:")
print(BASE)
print("\nOriginal images:")
print(ORIGINAL_ROOT)
print("\nAnnotations:")
print(ANNOTATION_ROOT)
print("\nEnhanced images:")
print(ENHANCED_ROOT)

assert ORIGINAL_ROOT.exists()
assert ANNOTATION_ROOT.exists()
assert ENHANCED_ROOT.exists()

print("\n✅ DATASET PATHS VERIFIED")

FASTER R-CNN + RESNET50-FPN EXPERIMENT
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
CUDA available: True
GPU: Tesla T4

Dataset root:
/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final

Original images:
/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final/vehicle_dataset/Images

Annotations:
/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final/vehicle_dataset/Annotations

Enhanced images:
/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final/enhanced_vehicle_dataset/Images

✅ DATASET PATHS VERIFIED


In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

print("=" * 80)
print("GENERATING FIXED 70/15/15 STRATIFIED SPLIT")
print("=" * 80)

CLASSES = [
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
]

VALID_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

rows = []

# ------------------------------------------------------------
# Build deterministic sorted manifest
# ------------------------------------------------------------
for class_name in CLASSES:
    image_dir = ORIGINAL_ROOT / class_name
    annotation_dir = ANNOTATION_ROOT / class_name
    enhanced_dir = ENHANCED_ROOT / class_name

    # Annotation lookup: example image.jpg -> image.jpg.txt
    annotation_lookup = {
        p.name[:-4].lower(): p
        for p in annotation_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    }

    image_files = sorted(
        [
            p for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VALID_EXTS
        ],
        key=lambda x: x.name.lower()
    )

    for image_path in image_files:
        key = image_path.name.lower()

        if key not in annotation_lookup:
            raise FileNotFoundError(
                f"Annotation missing for: {image_path}"
            )

        enhanced_path = enhanced_dir / image_path.name

        # Case-sensitive filesystem fallback
        if not enhanced_path.exists():
            enhanced_lookup = {
                p.name.lower(): p
                for p in enhanced_dir.iterdir()
                if p.is_file()
            }
            if key not in enhanced_lookup:
                raise FileNotFoundError(
                    f"Enhanced image missing: "
                    f"{class_name}/{image_path.name}"
                )
            enhanced_path = enhanced_lookup[key]

        annotation_path = annotation_lookup[key]

        rows.append({
            "Class": class_name,
            "Filename": image_path.name,
            "RelativePath": f"{class_name}/{image_path.name}",
            "AnnotationRelativePath": f"{class_name}/{annotation_path.name}"
        })

manifest = pd.DataFrame(rows)

print("\nTotal usable images:", len(manifest))
print("\nClass distribution:")
print(manifest["Class"].value_counts().reindex(CLASSES))

assert len(manifest) == 2997, (
    f"Expected 2997 images, found {len(manifest)}"
)

# ------------------------------------------------------------
# 70% Train + 30% Temporary
# ------------------------------------------------------------
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=42,
    stratify=manifest["Class"]
)

# ------------------------------------------------------------
# Split remaining 30% equally -> 15% Val + 15% Test
# ------------------------------------------------------------
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["Class"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["Split"] = "train"
val_df["Split"] = "val"
test_df["Split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("\nSplit sizes:")
print(split_df["Split"].value_counts())

print("\nClass x Split:")
summary = pd.crosstab(split_df["Class"], split_df["Split"])
summary = summary.reindex(CLASSES)
print(summary)

# Expected totals
assert len(train_df) == 2097
assert len(val_df) == 450
assert len(test_df) == 450

expected = {
    "Bicycle":   {"train": 456, "val": 97,  "test": 98},
    "Boat":      {"train": 475, "val": 102, "test": 102},
    "Bus":       {"train": 369, "val": 79,  "test": 79},
    "Car":       {"train": 446, "val": 96,  "test": 96},
    "Motorbike": {"train": 351, "val": 76,  "test": 75}
}

for cls in CLASSES:
    for split in ["train", "val", "test"]:
        actual = (
            split_df[
                (split_df["Class"] == cls) &
                (split_df["Split"] == split)
            ].shape[0]
        )
        assert actual == expected[cls][split], (
            f"Mismatch: {cls} {split}: "
            f"{actual} != {expected[cls][split]}"
        )

# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------
train_ids = set(train_df["RelativePath"])
val_ids = set(val_df["RelativePath"])
test_ids = set(test_df["RelativePath"])

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------
CSV_PATH = "/kaggle/working/vehicle_split_70_15_15.csv"
split_df.to_csv(CSV_PATH, index=False)

print("\nCSV saved:")
print(CSV_PATH)
print("\n✅ FIXED SPLIT CREATED SUCCESSFULLY")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

GENERATING FIXED 70/15/15 STRATIFIED SPLIT

Total usable images: 2997

Class distribution:
Class
Bicycle      651
Boat         679
Bus          527
Car          638
Motorbike    502
Name: count, dtype: int64

Split sizes:
Split
train    2097
val       450
test      450
Name: count, dtype: int64

Class x Split:
Split      test  train  val
Class                      
Bicycle      98    456   97
Boat        102    475  102
Bus          79    369   79
Car          96    446   96
Motorbike    75    351   76

CSV saved:
/kaggle/working/vehicle_split_70_15_15.csv

✅ FIXED SPLIT CREATED SUCCESSFULLY
Train: 2097
Validation: 450
Test: 450


In [3]:
import random
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision.transforms import functional as F

CLASS_TO_ID = {
    "Bicycle": 1,
    "Boat": 2,
    "Bus": 3,
    "Car": 4,
    "Motorbike": 5
}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}


class VehicleDetectionDataset(Dataset):
    def __init__(self, dataframe, image_root, annotation_root, training=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.annotation_root = Path(annotation_root)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.image_root / row["RelativePath"]
        annotation_path = self.annotation_root / row["AnnotationRelativePath"]

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []

        # ----------------------------------------------------
        # Read ExDark bbGt annotation
        # ----------------------------------------------------
        with open(annotation_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if not line:
                continue
            if line.startswith("%"):
                continue

            parts = line.split()
            if len(parts) < 5:
                continue

            class_name = parts[0]
            if class_name not in CLASS_TO_ID:
                continue

            x = float(parts[1])
            y = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            xmin = max(0.0, x)
            ymin = max(0.0, y)
            xmax = min(float(width), x + w)
            ymax = min(float(height), y + h)

            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_TO_ID[class_name])

        if len(boxes) == 0:
            raise RuntimeError(f"No valid boxes: {annotation_path}")

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        # ----------------------------------------------------
        # Convert image -> tensor [0,1]
        # ----------------------------------------------------
        image = F.to_tensor(image)

        # ----------------------------------------------------
        # Training augmentation: horizontal flip only
        # ----------------------------------------------------
        if self.training:
            if random.random() < 0.5:
                image = torch.flip(image, dims=[2])

                old_xmin = boxes[:, 0].clone()
                old_xmax = boxes[:, 2].clone()

                boxes[:, 0] = width - old_xmax
                boxes[:, 2] = width - old_xmin

        target = {
            "boxes": boxes,
            "labels": labels
        }

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


print("✅ DATASET CLASS READY")

✅ DATASET CLASS READY


In [4]:
from torch.utils.data import DataLoader

train_df = split_df[split_df["Split"] == "train"].reset_index(drop=True)
val_df = split_df[split_df["Split"] == "val"].reset_index(drop=True)
test_df = split_df[split_df["Split"] == "test"].reset_index(drop=True)

# ============================================================
# ORIGINAL
# ============================================================
original_train_dataset = VehicleDetectionDataset(
    train_df, ORIGINAL_ROOT, ANNOTATION_ROOT, training=True
)
original_val_dataset = VehicleDetectionDataset(
    val_df, ORIGINAL_ROOT, ANNOTATION_ROOT, training=False
)
original_test_dataset = VehicleDetectionDataset(
    test_df, ORIGINAL_ROOT, ANNOTATION_ROOT, training=False
)

# ============================================================
# ENHANCED — Same boxes, same IDs, same split
# ============================================================
enhanced_train_dataset = VehicleDetectionDataset(
    train_df, ENHANCED_ROOT, ANNOTATION_ROOT, training=True
)
enhanced_val_dataset = VehicleDetectionDataset(
    val_df, ENHANCED_ROOT, ANNOTATION_ROOT, training=False
)
enhanced_test_dataset = VehicleDetectionDataset(
    test_df, ENHANCED_ROOT, ANNOTATION_ROOT, training=False
)

# ------------------------------------------------------------
# Reproducible DataLoader generator
# ------------------------------------------------------------
generator = torch.Generator()
generator.manual_seed(42)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        num_workers=2,
        collate_fn=collate_fn,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=generator
    )


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn,
        pin_memory=True
    )


original_train_loader = make_train_loader(original_train_dataset)
original_val_loader = make_eval_loader(original_val_dataset)
original_test_loader = make_eval_loader(original_test_dataset)

enhanced_train_loader = make_train_loader(enhanced_train_dataset)
enhanced_val_loader = make_eval_loader(enhanced_val_dataset)
enhanced_test_loader = make_eval_loader(enhanced_test_dataset)

print("=" * 60)
print("DATASET SIZES")
print("=" * 60)
print("Original Train:", len(original_train_dataset))
print("Original Val:", len(original_val_dataset))
print("Original Test:", len(original_test_dataset))
print("Enhanced Train:", len(enhanced_train_dataset))
print("Enhanced Val:", len(enhanced_val_dataset))
print("Enhanced Test:", len(enhanced_test_dataset))

assert len(original_train_dataset) == 2097
assert len(original_val_dataset) == 450
assert len(original_test_dataset) == 450
assert len(enhanced_train_dataset) == 2097
assert len(enhanced_val_dataset) == 450
assert len(enhanced_test_dataset) == 450

print("\n✅ ORIGINAL/ENHANCED PAIRING READY")

DATASET SIZES
Original Train: 2097
Original Val: 450
Original Test: 450
Enhanced Train: 2097
Enhanced Val: 450
Enhanced Test: 450

✅ ORIGINAL/ENHANCED PAIRING READY


In [5]:
image, target = original_train_dataset[0]

print("Image tensor shape:", image.shape)
print("Image dtype:", image.dtype)
print("\nBoxes:")
print(target["boxes"])
print("\nLabels:")
print(target["labels"])
print("\nNumber of boxes:")
print(len(target["boxes"]))

assert image.ndim == 3
assert image.shape[0] == 3
assert len(target["boxes"]) > 0
assert len(target["boxes"]) == len(target["labels"])

print("\n✅ SANITY CHECK PASSED")

Image tensor shape: torch.Size([3, 640, 640])
Image dtype: torch.float32

Boxes:
tensor([[ 91., 399., 637., 584.]])

Labels:
tensor([3])

Number of boxes:
1

✅ SANITY CHECK PASSED


In [6]:
try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    print("torchmetrics available")
except Exception as e:
    print(e)
    print("If missing, run: !pip install -q torchmetrics pycocotools")

torchmetrics available


In [7]:
import os
import time
import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

NUM_CLASSES = 6  # Background + 5 vehicle classes


def create_model():
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(
        weights=weights,
        # Keep detector input scale controlled
        min_size=640,
        max_size=640
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

    return model


@torch.no_grad()
def evaluate_map(model, loader):
    model.eval()
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        class_metrics=True
    )

    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)

        preds_cpu = []
        targets_cpu = []

        for output, target in zip(outputs, targets):
            preds_cpu.append({
                "boxes": output["boxes"].detach().cpu(),
                "scores": output["scores"].detach().cpu(),
                "labels": output["labels"].detach().cpu()
            })
            targets_cpu.append({
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            })

        metric.update(preds_cpu, targets_cpu)

    result = metric.compute()
    return result


print("✅ MODEL AND EVALUATION FUNCTIONS READY")

✅ MODEL AND EVALUATION FUNCTIONS READY


In [8]:
def train_fasterrcnn(experiment_name, train_loader, val_loader):
    print("=" * 80)
    print(experiment_name)
    print("=" * 80)

    model = create_model().to(DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params, lr=0.005, momentum=0.9, weight_decay=0.0005
    )

    MAX_EPOCHS = 100
    PATIENCE = 15

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS
    )

    output_dir = Path(
        "/kaggle/working/vehicle_detection_final/" f"{experiment_name}"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    best_path = output_dir / "best.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"

    best_map = -1.0
    no_improvement = 0
    history = []

    training_start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.perf_counter()

        # ====================================================
        # TRAIN
        # ====================================================
        model.train()
        running_loss = 0.0
        num_batches = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = [image.to(DEVICE) for image in images]
            targets = [
                {k: v.to(DEVICE) for k, v in target.items()}
                for target in targets
            ]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            running_loss += losses.item()
            num_batches += 1

            if (batch_idx + 1) % 100 == 0:
                print(
                    f"Epoch {epoch:03d} | "
                    f"Batch {batch_idx + 1:04d}/{len(train_loader)} | "
                    f"Loss {losses.item():.4f}"
                )

        avg_train_loss = running_loss / max(num_batches, 1)

        # ====================================================
        # VALIDATION
        # ====================================================
        val_metrics = evaluate_map(model, val_loader)
        val_map50 = float(val_metrics["map_50"])
        val_map5095 = float(val_metrics["map"])

        scheduler.step()

        epoch_time = time.perf_counter() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"\nEpoch {epoch:03d} COMPLETE")
        print(f"Train Loss : {avg_train_loss:.4f}")
        print(f"Val mAP50 : {val_map50:.4f}")
        print(f"Val mAP50-95: {val_map5095:.4f}")
        print(f"LR : {current_lr:.6f}")
        print(f"Epoch time : {epoch_time / 60:.2f} min")

        # ====================================================
        # History
        # ====================================================
        history.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_map50": val_map50,
            "val_map50_95": val_map5095,
            "lr": current_lr,
            "epoch_minutes": epoch_time / 60
        })
        pd.DataFrame(history).to_csv(history_path, index=False)

        # ====================================================
        # Last checkpoint
        # ====================================================
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_map50": val_map50,
            "val_map50_95": val_map5095
        }, last_path)

        # ====================================================
        # Best checkpoint
        # ====================================================
        if val_map5095 > best_map:
            best_map = val_map5095
            no_improvement = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_map50": val_map50,
                "val_map50_95": val_map5095
            }, best_path)
            print("✅ New BEST model saved.")
        else:
            no_improvement += 1
            print("No improvement:", f"{no_improvement}/{PATIENCE}")

        print("-" * 80)

        # ====================================================
        # Early stopping
        # ====================================================
        if no_improvement >= PATIENCE:
            print("\nEARLY STOPPING TRIGGERED")
            break

    total_hours = (time.perf_counter() - training_start) / 3600

    print("\n" + "=" * 80)
    print(f"{experiment_name} COMPLETE")
    print(f"Best validation mAP50-95: {best_map:.4f}")
    print(f"Total training time: {total_hours:.3f} hours")
    print("Best checkpoint:", best_path)
    print("=" * 80)

    return best_path

In [9]:
ORIGINAL_BEST = train_fasterrcnn(
    experiment_name="fasterrcnn_resnet50_fpn_original_final",
    train_loader=original_train_loader,
    val_loader=original_val_loader
)

print("\nOriginal best checkpoint:")
print(ORIGINAL_BEST)

fasterrcnn_resnet50_fpn_original_final
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 238MB/s]


Epoch 001 | Batch 0100/1049 | Loss 0.5813
Epoch 001 | Batch 0200/1049 | Loss 0.1744
Epoch 001 | Batch 0300/1049 | Loss 0.3886
Epoch 001 | Batch 0400/1049 | Loss 0.2269
Epoch 001 | Batch 0500/1049 | Loss 0.3887
Epoch 001 | Batch 0600/1049 | Loss 0.4161
Epoch 001 | Batch 0700/1049 | Loss 0.5872
Epoch 001 | Batch 0800/1049 | Loss 0.2622
Epoch 001 | Batch 0900/1049 | Loss 0.0870
Epoch 001 | Batch 1000/1049 | Loss 0.8375

Epoch 001 COMPLETE
Train Loss : 0.3753
Val mAP50 : 0.7392
Val mAP50-95: 0.4303
LR : 0.004999
Epoch time : 5.39 min
✅ New BEST model saved.
--------------------------------------------------------------------------------
Epoch 002 | Batch 0100/1049 | Loss 0.1489
Epoch 002 | Batch 0200/1049 | Loss 0.0881
Epoch 002 | Batch 0300/1049 | Loss 0.2044
Epoch 002 | Batch 0400/1049 | Loss 0.2039
Epoch 002 | Batch 0500/1049 | Loss 0.1553
Epoch 002 | Batch 0600/1049 | Loss 0.1740
Epoch 002 | Batch 0700/1049 | Loss 0.5756
Epoch 002 | Batch 0800/1049 | Loss 0.5882
Epoch 002 | Batch 0900/

In [10]:
ENHANCED_BEST = train_fasterrcnn(
    experiment_name="fasterrcnn_resnet50_fpn_enhanced_final",
    train_loader=enhanced_train_loader,
    val_loader=enhanced_val_loader
)

print("\nEnhanced best checkpoint:")
print(ENHANCED_BEST)

fasterrcnn_resnet50_fpn_enhanced_final
Epoch 001 | Batch 0100/1049 | Loss 1.0114
Epoch 001 | Batch 0200/1049 | Loss 0.2968
Epoch 001 | Batch 0300/1049 | Loss 0.2337
Epoch 001 | Batch 0400/1049 | Loss 0.2416
Epoch 001 | Batch 0500/1049 | Loss 0.1381
Epoch 001 | Batch 0600/1049 | Loss 0.3747
Epoch 001 | Batch 0700/1049 | Loss 0.4414
Epoch 001 | Batch 0800/1049 | Loss 0.2833
Epoch 001 | Batch 0900/1049 | Loss 0.6080
Epoch 001 | Batch 1000/1049 | Loss 0.1854

Epoch 001 COMPLETE
Train Loss : 0.3738
Val mAP50 : 0.7461
Val mAP50-95: 0.4251
LR : 0.004999
Epoch time : 5.47 min
✅ New BEST model saved.
--------------------------------------------------------------------------------
Epoch 002 | Batch 0100/1049 | Loss 0.1674
Epoch 002 | Batch 0200/1049 | Loss 0.2763
Epoch 002 | Batch 0300/1049 | Loss 1.0055
Epoch 002 | Batch 0400/1049 | Loss 0.4960
Epoch 002 | Batch 0500/1049 | Loss 0.3063
Epoch 002 | Batch 0600/1049 | Loss 0.3138
Epoch 002 | Batch 0700/1049 | Loss 0.1497
Epoch 002 | Batch 0800/104

In [11]:
from torchvision.ops import box_iou


def load_trained_model(checkpoint_path):
    model = create_model()
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


@torch.no_grad()
def full_test_evaluation(model, loader, experiment_name):
    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)

    model.eval()

    # --------------------------------------------------------
    # COCO-style mAP
    # --------------------------------------------------------
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        class_metrics=True
    )

    detection_records = []
    total_gt = 0

    for images, targets in loader:
        images_gpu = [img.to(DEVICE) for img in images]
        outputs = model(images_gpu)

        for output, target in zip(outputs, targets):
            pred_boxes = output["boxes"].detach().cpu()
            pred_scores = output["scores"].detach().cpu()
            pred_labels = output["labels"].detach().cpu()

            gt_boxes = target["boxes"].cpu()
            gt_labels = target["labels"].cpu()

            metric.update(
                [{"boxes": pred_boxes, "scores": pred_scores, "labels": pred_labels}],
                [{"boxes": gt_boxes, "labels": gt_labels}]
            )

            total_gt += len(gt_boxes)

            # ------------------------------------------------
            # Greedy TP/FP matching at IoU 0.50
            # Used to obtain P/R/F1 at best confidence point
            # ------------------------------------------------
            for class_id in range(1, NUM_CLASSES):
                p_mask = (pred_labels == class_id)
                g_mask = (gt_labels == class_id)

                p_boxes = pred_boxes[p_mask]
                p_scores = pred_scores[p_mask]
                g_boxes = gt_boxes[g_mask]

                if len(p_boxes) == 0:
                    continue

                order = torch.argsort(p_scores, descending=True)
                p_boxes = p_boxes[order]
                p_scores = p_scores[order]

                matched_gt = set()

                if len(g_boxes) > 0:
                    ious = box_iou(p_boxes, g_boxes)
                else:
                    ious = torch.empty((len(p_boxes), 0))

                for pred_idx in range(len(p_boxes)):
                    is_tp = 0

                    if len(g_boxes) > 0:
                        values = ious[pred_idx]
                        best_iou, best_gt = torch.max(values, dim=0)
                        best_gt = int(best_gt.item())

                        if (best_iou.item() >= 0.50 and best_gt not in matched_gt):
                            is_tp = 1
                            matched_gt.add(best_gt)

                    detection_records.append(
                        (float(p_scores[pred_idx]), is_tp)
                    )

    result = metric.compute()
    map50 = float(result["map_50"])
    map5095 = float(result["map"])

    # --------------------------------------------------------
    # Precision / Recall / F1 curve
    # --------------------------------------------------------
    detection_records.sort(key=lambda x: x[0], reverse=True)

    scores = np.array([x[0] for x in detection_records])
    tp = np.array([x[1] for x in detection_records], dtype=np.float64)
    fp = 1.0 - tp

    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)

    precision_curve = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
    recall_curve = cum_tp / max(total_gt, 1)

    f1_curve = (
        2 * precision_curve * recall_curve /
        np.maximum(precision_curve + recall_curve, 1e-12)
    )

    best_index = int(np.argmax(f1_curve))

    precision = float(precision_curve[best_index])
    recall = float(recall_curve[best_index])
    f1 = float(f1_curve[best_index])
    best_conf = float(scores[best_index])

    print(f"Precision : {precision:.6f}")
    print(f"Recall : {recall:.6f}")
    print(f"F1 : {f1:.6f}")
    print(f"mAP@50 : {map50:.6f}")
    print(f"mAP@50:95 : {map5095:.6f}")
    print(f"Best confidence : {best_conf:.6f}")
    print(f"Ground-truth boxes: {total_gt}")

    return {
        "Condition": experiment_name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "mAP50": map50,
        "mAP50_95": map5095,
        "BestConfidence": best_conf
    }


# ============================================================
# LOAD BOTH BEST MODELS
# ============================================================
original_model = load_trained_model(ORIGINAL_BEST)
enhanced_model = load_trained_model(ENHANCED_BEST)

# ============================================================
# 2 × 2 TEST MATRIX
# ============================================================
result_OO = full_test_evaluation(
    original_model, original_test_loader, "Original-trained -> Original Test"
)
result_OE = full_test_evaluation(
    original_model, enhanced_test_loader, "Original-trained -> Enhanced Test"
)
result_EO = full_test_evaluation(
    enhanced_model, original_test_loader, "Enhanced-trained -> Original Test"
)
result_EE = full_test_evaluation(
    enhanced_model, enhanced_test_loader, "Enhanced-trained -> Enhanced Test"
)

results_df = pd.DataFrame([result_OO, result_OE, result_EO, result_EE])

print("\n" + "=" * 100)
print("FASTER R-CNN + RESNET50-FPN TEST RESULTS")
print("=" * 100)
display(results_df)

results_df.to_csv(
    "/kaggle/working/fasterrcnn_resnet50_fpn_2x2_metrics.csv",
    index=False
)

print("\n✅ 2x2 test evaluation complete.")


Original-trained -> Original Test
Precision : 0.806061
Recall : 0.688406
F1 : 0.742602
mAP@50 : 0.776043
mAP@50:95 : 0.475552
Best confidence : 0.728055
Ground-truth boxes: 966

Original-trained -> Enhanced Test
Precision : 0.830417
Recall : 0.638716
F1 : 0.722060
mAP@50 : 0.744674
mAP@50:95 : 0.454836
Best confidence : 0.789184
Ground-truth boxes: 966

Enhanced-trained -> Original Test
Precision : 0.724138
Recall : 0.652174
F1 : 0.686275
mAP@50 : 0.713593
mAP@50:95 : 0.436119
Best confidence : 0.659792
Ground-truth boxes: 966

Enhanced-trained -> Enhanced Test
Precision : 0.822985
Recall : 0.644928
F1 : 0.723157
mAP@50 : 0.758736
mAP@50:95 : 0.454894
Best confidence : 0.832861
Ground-truth boxes: 966

FASTER R-CNN + RESNET50-FPN TEST RESULTS


,Condition,Precision,Recall,F1,mAP50,mAP50_95,BestConfidence
0,Original-trained -> Original Test,0.806061,0.688406,0.742602,0.776043,0.475552,0.728055
1,Original-trained -> Enhanced Test,0.830417,0.638716,0.722060,0.744674,0.454836,0.789184
2,Enhanced-trained -> Original Test,0.724138,0.652174,0.686275,0.713593,0.436119,0.659792
3,Enhanced-trained -> Enhanced Test,0.822985,0.644928,0.723157,0.758736,0.454894,0.832861



✅ 2x2 test evaluation complete.


In [12]:
import time
from PIL import Image
from torchvision.transforms import functional as F


@torch.no_grad()
def benchmark_fasterrcnn(model, dataframe, image_root, experiment_name):
    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)

    model.eval()

    image_paths = [
        Path(image_root) / rel_path
        for rel_path in dataframe["RelativePath"]
    ]

    print("Test images found:", len(image_paths))
    assert len(image_paths) == 450

    # --------------------------------------------------------
    # WARMUP
    # --------------------------------------------------------
    print("Running 20-image warm-up...")
    for path in image_paths[:20]:
        image = Image.open(path).convert("RGB")
        tensor = F.to_tensor(image).to(DEVICE)
        _ = model([tensor])

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Warm-up complete.")

    # --------------------------------------------------------
    # TIMING
    # Includes image loading + tensor conversion +
    # detector preprocessing + inference + postprocessing
    # --------------------------------------------------------
    times = []
    print("Benchmarking all 450 images...")

    for path in image_paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()

        image = Image.open(path).convert("RGB")
        tensor = F.to_tensor(image).to(DEVICE)
        _ = model([tensor])

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()

        times.append(end - start)

    times = np.asarray(times)
    total_time = times.sum()
    mean_latency = times.mean()
    median_latency = np.median(times)
    fps = len(times) / total_time

    print(f"Images tested : {len(times)}")
    print(f"Total time : {total_time:.3f} s")
    print(f"Mean latency : {mean_latency*1000:.3f} ms/image")
    print(f"Median latency : {median_latency*1000:.3f} ms/image")
    print(f"Overall throughput FPS : {fps:.2f}")
    print(f"Min latency : {times.min()*1000:.3f} ms")
    print(f"Max latency : {times.max()*1000:.3f} ms")

    return {
        "Condition": experiment_name,
        "MeanLatency_ms": mean_latency * 1000,
        "MedianLatency_ms": median_latency * 1000,
        "FPS": fps
    }


fps_OO = benchmark_fasterrcnn(
    original_model, test_df, ORIGINAL_ROOT, "Original-trained -> Original Test"
)
fps_OE = benchmark_fasterrcnn(
    original_model, test_df, ENHANCED_ROOT, "Original-trained -> Enhanced Test"
)
fps_EO = benchmark_fasterrcnn(
    enhanced_model, test_df, ORIGINAL_ROOT, "Enhanced-trained -> Original Test"
)
fps_EE = benchmark_fasterrcnn(
    enhanced_model, test_df, ENHANCED_ROOT, "Enhanced-trained -> Enhanced Test"
)

fps_df = pd.DataFrame([fps_OO, fps_OE, fps_EO, fps_EE])

print("\n" + "=" * 100)
print("CONTROLLED FPS SUMMARY")
print("=" * 100)
display(fps_df)

fps_df.to_csv("/kaggle/working/fasterrcnn_resnet50_fpn_fps.csv", index=False)

print("\n✅ FPS BENCHMARK COMPLETE")


Original-trained -> Original Test
Test images found: 450
Running 20-image warm-up...
Warm-up complete.
Benchmarking all 450 images...
Images tested : 450
Total time : 31.774 s
Mean latency : 70.610 ms/image
Median latency : 60.130 ms/image
Overall throughput FPS : 14.16
Min latency : 43.764 ms
Max latency : 361.979 ms

Original-trained -> Enhanced Test
Test images found: 450
Running 20-image warm-up...
Warm-up complete.
Benchmarking all 450 images...
Images tested : 450
Total time : 31.846 s
Mean latency : 70.769 ms/image
Median latency : 60.394 ms/image
Overall throughput FPS : 14.13
Min latency : 46.587 ms
Max latency : 428.256 ms

Enhanced-trained -> Original Test
Test images found: 450
Running 20-image warm-up...
Warm-up complete.
Benchmarking all 450 images...
Images tested : 450
Total time : 31.753 s
Mean latency : 70.562 ms/image
Median latency : 59.954 ms/image
Overall throughput FPS : 14.17
Min latency : 45.989 ms
Max latency : 379.633 ms

Enhanced-trained -> Enhanced Test
Te

,Condition,MeanLatency_ms,MedianLatency_ms,FPS
0,Original-trained -> Original Test,70.609598,60.129756,14.162381
1,Original-trained -> Enhanced Test,70.768714,60.394313,14.130538
2,Enhanced-trained -> Original Test,70.561570,59.954364,14.172020
3,Enhanced-trained -> Enhanced Test,69.832362,59.598593,14.320008



✅ FPS BENCHMARK COMPLETE


In [13]:
FINAL_RESULTS = results_df.merge(
    fps_df[["Condition", "MeanLatency_ms", "MedianLatency_ms", "FPS"]],
    on="Condition",
    how="left"
)

FINAL_RESULTS[
    ["Precision", "Recall", "F1", "mAP50", "mAP50_95"]
] *= 100

print("=" * 120)
print("FASTER R-CNN + RESNET50-FPN FINAL 2x2 RESULTS")
print("=" * 120)
display(FINAL_RESULTS.round(3))

FINAL_RESULTS.to_csv(
    "/kaggle/working/fasterrcnn_resnet50_fpn_FINAL_RESULTS.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/fasterrcnn_resnet50_fpn_FINAL_RESULTS.csv")
print("\n✅ EXPERIMENT COMPLETE")

FASTER R-CNN + RESNET50-FPN FINAL 2x2 RESULTS


,Condition,Precision,Recall,F1,mAP50,mAP50_95,BestConfidence,MeanLatency_ms,MedianLatency_ms,FPS
0,Original-trained -> Original Test,80.606,68.841,74.260,77.604,47.555,0.728,70.610,60.130,14.162
1,Original-trained -> Enhanced Test,83.042,63.872,72.206,74.467,45.484,0.789,70.769,60.394,14.131
2,Enhanced-trained -> Original Test,72.414,65.217,68.627,71.359,43.612,0.660,70.562,59.954,14.172
3,Enhanced-trained -> Enhanced Test,82.299,64.493,72.316,75.874,45.489,0.833,69.832,59.599,14.320



Saved:
/kaggle/working/fasterrcnn_resnet50_fpn_FINAL_RESULTS.csv

✅ EXPERIMENT COMPLETE
